<a href="https://colab.research.google.com/github/kumarsirish/-RAG_legal_docs/blob/main/RAG_Assg_Legal_Documents_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Extracting Information from Legal Documents Using RAG**

## **Objective**

The main objective of this assignment is to process and analyse a collection text files containing legal agreements (e.g., NDAs) to prepare them for implementing a **Retrieval-Augmented Generation (RAG)** system. This involves:

* Understand the Cleaned Data : Gain a comprehensive understanding of the structure, content, and context of the cleaned dataset.
* Perform Exploratory Analysis : Conduct bivariate and multivariate analyses to uncover relationships and trends within the cleaned data.
* Create Visualisations : Develop meaningful visualisations to support the analysis and make findings interpretable.
* Derive Insights and Conclusions : Extract valuable insights from the cleaned data and provide clear, actionable conclusions.
* Document the Process : Provide a detailed description of the data, its attributes, and the steps taken during the analysis for reproducibility and clarity.

The ultimate goal is to transform the raw text data into a clean, structured, and analysable format that can be effectively used to build and train a RAG system for tasks like information retrieval, question-answering, and knowledge extraction related to legal agreements.

### **Business Value**  


The project aims to leverage RAG to enhance legal document processing for businesses, law firms, and regulatory bodies. The key business objectives include:

* Faster Legal Research: <br> Reduce the time lawyers and compliance officers spend searching for relevant case laws, precedents, statutes, or contract clauses.
* Improved Contract Analysis: <br> Automatically extract key terms, obligations, and risks from lengthy contracts.
* Regulatory Compliance Monitoring: <br> Help businesses stay updated with legal and regulatory changes by retrieving relevant legal updates.
* Enhanced Decision-Making: <br> Provide accurate and context-aware legal insights to assist in risk assessment and legal strategy.


**Use Cases**
* Legal Chatbots
* Contract Review Automation
* Tracking Regulatory Changes and Compliance Monitoring
* Case Law Analysis of past judgments
* Due Diligence & Risk Assessment

## **1. Data Loading, Preparation and Analysis** <font color=red> [20 marks] </font><br>

### **1.1 Data Understanding**

The dataset contains legal documents and contracts collected from various sources. The documents are present as text files (`.txt`) in the *corpus* folder.

There are four types of documents in the *courpus* folder, divided into four subfolders.
- `contractnli`: contains various non-disclosure and confidentiality agreements
- `cuad`: contains contracts with annotated legal clauses
- `maud`: contains various merger/acquisition contracts and agreements
- `privacy_qa`: a question-answering dataset containing privacy policies

The dataset also contains evaluation files in JSON format in the *benchmark* folder. The files contain the questions and their answers, along with sources. For the above folders, there is a `json` file: `contractnli.json`, `cuad.json`, `maud.json`. The file structure is as follows:

```
{
    "tests": [
        {
            "query": <question1>,
            "snippets": [{
                    "file_path": <source_file1>,
                    "span": [ begin_position, end_position ],
                    "answer": <relevant answer to the question 1>
                },
                {
                    "file_path": <source_file2>,
                    "span": [ begin_position, end_position ],
                    "answer": <relevant answer to the question 2>
                }, ....
            ]
        },
        {
            "query": <question2>,
            "snippets": [{<answer context for que 2>}]
        },
        ... <more queries>
    ]
}
```

### **1.2 Load and Preprocess the data** <font color=red> [5 marks] </font><br>

#### Loading libraries

In [1]:
## The following libraries might be useful
#%pip install -q langchain-openai
#%pip install -U -q langchain-community
#%pip install -U -q langchain-chroma
#%pip install -U -q datasets pyarrow pandas
#%pip install -U -q ragas
#%pip install -U -q rouge_score
#%pip install -q sentence-transformers qdrant-client
!rm -f requirements.txt
!wget -O requirements.txt https://raw.githubusercontent.com/kumarsirish/-RAG_legal_docs/main/requirements.txt
!pip install -r requirements.txt

--2026-06-01 15:24:35--  https://raw.githubusercontent.com/kumarsirish/-RAG_legal_docs/main/requirements.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 255 [text/plain]
Saving to: ‘requirements.txt’

requirements.txt    100%[===================>]     255  --.-KB/s    in 0s      

2026-06-01 15:24:35 (16.8 MB/s) - ‘requirements.txt’ saved [255/255]



In [3]:
!pip freeze
# Import essential libraries



absl-py==1.4.0
accelerate==1.13.0
access==1.1.10.post3
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.2
aiohttp==3.13.5
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.18.4
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.13.0
anywidget==0.9.21
appdirs==1.4.4
apsw==3.53.1.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2026.5.18.1.11.28
astunparse==1.6.3
atpublic==5.1
attrs==26.1.0
audioread==3.1.0
Authlib==1.7.2
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
bcrypt==5.0.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.40.0
bigquery-magics==0.14.0
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.3.3
bokeh==3.8.2
Bottleneck==1.4.2
bqplot==0.12.47
branca==0.8.2
brotli==1.2.0
build==1.5.0
CacheControl==0.14.4
cachetools==6.

#### **1.2.1** <font color=red> [3 marks] </font>
Load all `.txt` files from the folders.

You can utilise document loaders from the options provided by the LangChain community.

Optionally, you can also read the files manually, while ensuring proper handling of encoding issues (e.g., utf-8, latin1). In such case, also store the file content along with metadata (e.g., file name, directory path) for traceability.

In [5]:
import os
from langchain_core.documents import Document

# Updated URLs for direct download
corpus_zip_url = "https://github.com/kumarsirish/-RAG_legal_docs/raw/main/corpus.zip"
benchmark_zip_url = "https://github.com/kumarsirish/-RAG_legal_docs/raw/main/benchmark.zip"


# 1. Download and Unzip Corpus
!rm -rf corpus corpus.zip
!wget -O corpus.zip "{corpus_zip_url}"
!unzip -q -o corpus.zip -d .

# 2. Download and Unzip benchmarks
!rm -rf benchmark* benchmark.zip
!wget -O benchmark.zip "{benchmark_zip_url}"
!unzip -q -o benchmark.zip -d .

# Load documents from corpus
corpus_path = './corpus'
langchain_documents = []
documents = []

if os.path.exists(corpus_path):
    for root, _, files in os.walk(corpus_path):
        for file_name in files:
            if file_name.endswith('.txt'):
                file_path = os.path.join(root, file_name)
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                    langchain_documents.append(Document(page_content=content, metadata={'source': file_path}))
                except Exception as e:
                    print(f"Could not read file {file_path}: {e}")
    documents = langchain_documents
    print(f'Number of documents loaded: {len(documents)}')
else:
    print(f"Error: {corpus_path} not found.")

--2026-06-01 15:26:37--  https://github.com/kumarsirish/-RAG_legal_docs/raw/main/corpus.zip
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/kumarsirish/-RAG_legal_docs/main/corpus.zip [following]
--2026-06-01 15:26:38--  https://raw.githubusercontent.com/kumarsirish/-RAG_legal_docs/main/corpus.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16494949 (16M) [application/zip]
Saving to: ‘corpus.zip’

corpus.zip          100%[===================>]  15.73M  --.-KB/s    in 0.1s    

2026-06-01 15:26:38 (123 MB/s) - ‘corpus.zip’ saved [16494949/16494949]

--2026-06-01 15:26:40--  https://github.com

#### **1.2.2** <font color=red> [2 marks] </font>
Preprocess the text data to remove noise and prepare it for analysis.

Remove special characters, extra whitespace, and irrelevant content such as email and telephone contact info.
Normalise text (e.g., convert to lowercase, remove stop words).
Handle missing or corrupted data by logging errors and skipping problematic files.

In [6]:
# Clean and preprocess the data
# remove emails from the documents
!pip install nltk
import re
import nltk
nltk.download('stopwords')

def remove_emails(text):
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    return re.sub(email_pattern, '', text)
# remove phone numbers from the documents
def remove_phone_numbers(text):
    phone_pattern = r'\+?\d[\d -]{8,}\d'
    return re.sub(phone_pattern, '', text)
# remove special characters from the documents
def remove_special_characters(text):
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

#remove extra spaces from the documents
def remove_extra_spaces(text):
    return re.sub(r"[ \t]+", " ", text).strip()

#convert the documents to lowercase
def convert_to_lowercase(text):
    return text.lower()

#remove stop words from the documents
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
def remove_stop_words(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

# apply the cleaning functions to the documents
for doc in documents:
    doc.page_content = remove_emails(doc.page_content)
    doc.page_content = remove_phone_numbers(doc.page_content)
    doc.page_content = remove_special_characters(doc.page_content)
    doc.page_content = remove_extra_spaces(doc.page_content)
    doc.page_content = convert_to_lowercase(doc.page_content)
    doc.page_content = remove_stop_words(doc.page_content)
# Check the cleaned document
print(f'Cleaned example document: {documents[0].page_content[:500]}...')






[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Cleaned example document: nondisclosure confidentiality agreement partner ireland insurance limited individuals submitting information via ehandshake partnerrecom parties 1 partner ireland insurance limited 5th floor block 1 oval 160 shelbourne road dublin 4 ireland registered ireland company number 395190 2 individuals submitting information via interactive ehandshake partnerrecom party collectively referred parties background order allow parties engage insurance mediation activities business discussion parties agreed e...


### **1.3 Exploratory Data Analysis** <font color=red> [10 marks] </font><br>

#### **1.3.1** <font color=red> [2 marks] </font>
Calculate the average, maximum and minimum document length.

In [7]:
# Calculate the average, maximum and minimum document length.
doc_lengths = []  #count of words in each document

print(f'Number of documents: {len(documents)}')
for doc in documents:
    words = doc.page_content.split()
    length = len(words)
    doc_lengths.append(length)


average_length = sum(doc_lengths) / len(documents)
max_length = max(doc_lengths)
min_length = min(doc_lengths)
print(f'Average document length: {average_length}')
print(f'Maximum document length: {max_length}')
print(f'Minimum document length: {min_length}')

Number of documents: 645
Average document length: 8493.525581395348
Maximum document length: 84946
Minimum document length: 142


#### **1.3.2** <font color=red> [4 marks] </font>
Analyse the frequency of occurence of words and find the most and least occuring words.

Find the 20 most common and least common words in the text. Ignore stop words such as articles and prepositions.

In [8]:
# Find frequency of occurence of words
from collections import Counter
import pprint
word_freq = Counter()
for doc in documents:
    words = doc.page_content.split()
    #ignore single letter words
    words = [word for word in words if len(word) > 1]
    #ignore prepositions, conjunctions, articles etc. (stop words)
    words = [word for word in words if word not in stop_words]
    #shall and may are very common in legal documents, so ignore them as well
    words = [word for word in words if word not in ['shall', 'may']]
    #ignore roman numerals
    words = [word for word in words if not re.match(r'^[ivxlcdm]+$', word)]
    word_freq.update(words)
# Print the 10 most common words and their frequencies
pprint.pprint(word_freq.most_common(20))



[('company', 128587),
 ('agreement', 92881),
 ('section', 65804),
 ('parent', 52482),
 ('party', 44661),
 ('date', 34491),
 ('time', 30932),
 ('merger', 29777),
 ('material', 29413),
 ('subsidiaries', 28986),
 ('applicable', 27367),
 ('including', 25820),
 ('respect', 25405),
 ('stock', 22642),
 ('information', 22549),
 ('parties', 21881),
 ('business', 20671),
 ('prior', 20620),
 ('effective', 18704),
 ('required', 18619)]


#### **1.3.3** <font color=red> [4 marks] </font>
Analyse the similarity of different documents to each other based on TF-IDF vectors.

Transform some documents to TF-IDF vectors and calculate their similarity matrix using a suitable distance function. If contracts contain duplicate or highly similar clauses, similarity calculation can help detect them.

Identify for the first 10 documents and then for 10 random documents. What do you observe?

In [9]:
# Transform the page contents of documents - Modularized approach
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import random

def compute_tfidf_vectors(documents_list):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents_list)
    return vectorizer, tfidf_matrix

def compute_pairwise_similarity(tfidf_matrix):
    similarity_matrix = cosine_similarity(tfidf_matrix)
    return similarity_matrix

def analyze_similarity_pairwise(doc_contents, label, doc_indices=None):
    print(f"\n=== {label} ===")
    vectorizer, tfidf_matrix = compute_tfidf_vectors(doc_contents)

    # Compute pairwise similarity matrix
    similarity_matrix = compute_pairwise_similarity(tfidf_matrix)

    # Create labels for rows/columns
    if doc_indices is None:
        labels = [f'Doc {i}' for i in range(len(doc_contents))]
    else:
        labels = [f'Doc {idx}' for idx in doc_indices]

    # Create DataFrame for better visualization
    sim_df = pd.DataFrame(similarity_matrix, index=labels, columns=labels)
    print("\nPairwise Similarity Matrix:")
    print(sim_df.round(4))

    # Print statistics
    # Get upper triangle values (excluding diagonal)
    mask = np.triu(np.ones_like(similarity_matrix, dtype=bool), k=1)
    upper_triangle = similarity_matrix[mask]

    print(f"\nSimilarity Statistics:")
    print(f"  Average similarity: {upper_triangle.mean():.4f}")
    print(f"  Max similarity: {upper_triangle.max():.4f}")
    print(f"  Min similarity: {upper_triangle.min():.4f}")
    print(f"  Std deviation: {upper_triangle.std():.4f}")

# Analyze first 10 documents (same subdirectory)
doc_contents_first10 = [doc.page_content for doc in documents[:10]]
analyze_similarity_pairwise(doc_contents_first10, "FIRST 10 DOCUMENTS (Same Subdirectory)")

# Analyze 10 random documents (mixed subdirectories)
random_indices = random.sample(range(len(documents)), 10)
doc_contents_random = [documents[idx].page_content for idx in random_indices]
analyze_similarity_pairwise(doc_contents_random, "10 RANDOM DOCUMENTS (Mixed Subdirectories)", random_indices)


=== FIRST 10 DOCUMENTS (Same Subdirectory) ===

Pairwise Similarity Matrix:
        Doc 0   Doc 1   Doc 2   Doc 3   Doc 4   Doc 5   Doc 6   Doc 7   Doc 8  \
Doc 0  1.0000  0.0716  0.4950  0.3476  0.5453  0.4431  0.4809  0.2677  0.3615   
Doc 1  0.0716  1.0000  0.0951  0.0467  0.0697  0.0719  0.0873  0.0892  0.0516   
Doc 2  0.4950  0.0951  1.0000  0.2969  0.5175  0.5081  0.5178  0.2648  0.3216   
Doc 3  0.3476  0.0467  0.2969  1.0000  0.3901  0.3292  0.2764  0.3284  0.1922   
Doc 4  0.5453  0.0697  0.5175  0.3901  1.0000  0.5446  0.4848  0.2621  0.3672   
Doc 5  0.4431  0.0719  0.5081  0.3292  0.5446  1.0000  0.4785  0.2296  0.2970   
Doc 6  0.4809  0.0873  0.5178  0.2764  0.4848  0.4785  1.0000  0.2667  0.3320   
Doc 7  0.2677  0.0892  0.2648  0.3284  0.2621  0.2296  0.2667  1.0000  0.1812   
Doc 8  0.3615  0.0516  0.3216  0.1922  0.3672  0.2970  0.3320  0.1812  1.0000   
Doc 9  0.5725  0.1242  0.6324  0.3792  0.6121  0.5701  0.5758  0.3853  0.3791   

        Doc 9  
Doc 0  0.5725  

In [ ]:
# create a list of 10 random integers



In [ ]:
# Compute similarity scores for 10 random documents



### **1.4 Document Creation and Chunking** <font color=red> [5 marks] </font><br>

#### **1.4.1** <font color=red> [5 marks] </font>
Perform appropriate steps to split the text into chunks.

In [10]:
# Process files and generate chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = []
for doc in documents:
    doc_chunks = text_splitter.split_text(doc.page_content)
    chunks.extend(doc_chunks)
print(f'Number of chunks generated: {len(chunks)}')
print(f'Example chunk: {chunks[0][:500]}...')

Number of chunks generated: 55359
Example chunk: nondisclosure confidentiality agreement partner ireland insurance limited individuals submitting information via ehandshake partnerrecom parties 1 partner ireland insurance limited 5th floor block 1 oval 160 shelbourne road dublin 4 ireland registered ireland company number 395190 2 individuals submitting information via interactive ehandshake partnerrecom party collectively referred parties background order allow parties engage insurance mediation activities business discussion parties agreed e...


## **2. Vector Database and RAG Chain Creation** <font color=red> [15 marks] </font><br>

### **2.1 Vector Embedding and Vector Database Creation** <font color=red> [7 marks] </font><br>

#### **2.1.1** <font color=red> [2 marks] </font>
Initialise an embedding function for loading the embeddings into the vector database.

Initialise a function to transform the text to vectors using an embedding model. You can also use this function to transform during vector DB creation itself.

In [11]:

from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

model_name = "BAAI/bge-small-en-v1.5"   # or "all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_function = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": device}
)

print("Loaded:", model_name)
print("Device:", device)


/tmp/ipykernel_2082/2883210699.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded: BAAI/bge-small-en-v1.5
Device: cuda


#### Set up Qdrant Host and API Key

To connect to your Qdrant instance, you'll need the `QDRANT_HOST` and `QDRANT_API_KEY`. Please add these to Colab's secrets manager (the 🔑 icon in the left panel) and name them `QDRANT_HOST` and `QDRANT_API_KEY` respectively.

In [12]:
from google.colab import userdata

QDRANT_HOST = userdata.get('QDRANT_HOST')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
print("Qdrant credentials loaded from secrets.")

Qdrant credentials loaded from secrets.


In [13]:
# Initialise an embedding function



#### **2.1.2** <font color=red> [5 marks] </font>
Load the embeddings to a vector database.

Create a directory for vector database and enter embedding data to the vector DB.

In [14]:
import qdrant_client
from langchain_qdrant import QdrantVectorStore

# Initialize Qdrant Client
client = qdrant_client.QdrantClient(
    url=QDRANT_HOST,
    api_key=QDRANT_API_KEY
)

collection_name = "legal_docs"

# Check if collection exists
collections = client.get_collections().collections
collection_exists = any(c.name == collection_name for c in collections)

if not collection_exists:
    print(f"Collection '{collection_name}' not found. Creating and populating...")
    vector_store = QdrantVectorStore.from_texts(
        texts=chunks,
        embedding=embedding_function,
        url=QDRANT_HOST,
        api_key=QDRANT_API_KEY,
        collection_name=collection_name
    )
    print(f"Added {len(chunks)} chunks to Qdrant.")
else:
    print(f"Collection '{collection_name}' already exists. Connecting to existing store.")
    # Fixed: Added embedding parameter to allow dense retrieval
    vector_store = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embedding_function
    )

Collection 'legal_docs' already exists. Connecting to existing store.


### **2.2 Create RAG Chain** <font color=red> [8 marks] </font><br>

#### **2.2.1** <font color=red> [5 marks] </font>
Form the complete RAG pipeline.

You can either create a chain or directly the pipeline

In [15]:
import os
# 1. FIX: Force-upgrade the LangChain ecosystem to ensure compatibility.
# We use the latest stable releases for the 0.3.x line.
#%pip install -q -U langchain-openai langchain-core langchain-community

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

try:
    # 2. Initialize the LLM
    # Note: Ensure OPENAI_API_KEY is defined in your environment
    llm = ChatOpenAI(model="gpt-4o", temperature=0, api_key=OPENAI_API_KEY)

    # 3. Set up retriever
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    # 4. Prompt Template
    template = """You are an expert legal assistant. Answer the user's question using only the provided context below.
If you do not know the answer or if it's not explicitly mentioned in the context, say that you don't know. Do not make things up.

Context:
{context}

Question:
{question}

Answer:"""

    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # 5. Create the RAG chain
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    print("RAG chain successfully re-initialized.")

except ImportError as e:
    print(f"\nIMPORT ERROR: {e}")
    print("\n*** MANDATORY ACTION ***")
    print("1. Go to the menu: Runtime -> Restart session")
    print("2. Run THIS cell again immediately after restarting.")
except Exception as e:
    print(f"An error occurred: {e}")
    print("Ensure that 'vector_store' and 'OPENAI_API_KEY' are defined in previous cells.")

RAG chain successfully re-initialized.


#### **2.2.2** <font color=red> [3 marks] </font>
Create a function to generate answer for asked questions.

Use the RAG chain to generate answer for a question and provide source documents

In [16]:
# Create a function for question answering
question = "What are the termination obligations under the NDA?"
response = rag_chain.invoke(question)

print(response)



I don't know. The provided context does not explicitly mention the termination obligations under the NDA.


In [17]:
# Example question
question ="Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?"
response = rag_chain.invoke(question)
print(response)


I don't know.


## **3. RAG Evaluation** <font color=red> [10 marks] </font><br>

### **3.1 Evaluation and Inference** <font color=red> [10 marks] </font><br>

#### **3.1.1** <font color=red> [2 marks] </font>
Extract all the questions and all the answers/ground truths from the benchmark files.

Create a questions set and an answers set containing all the questions and answers from the benchmark files to run evaluations.

In [18]:
# Create a question set by taking all the questions from the benchmark data
# Also create a ground truth/answer set
import json
import os
import random

benchmark_dir = "./benchmarks"  # adjust if your path differs
benchmark_files = ["contractnli.json", "cuad.json", "maud.json"]

all_questions = []
all_answers = []

for filename in benchmark_files:
    filepath = os.path.join(benchmark_dir, filename)
    if not os.path.exists(filepath):
        print(f"Warning: {filepath} not found, skipping.")
        continue

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    for test in data.get("tests", []):
        query = test.get("query", "").strip()
        snippets = test.get("snippets", [])

        # Collect all answer texts for this question as a combined ground truth
        ground_truth_parts = [s.get("answer", "").strip() for s in snippets if s.get("answer", "").strip()]
        ground_truth = " ".join(ground_truth_parts)

        if query and ground_truth:
            all_questions.append(query)
            all_answers.append(ground_truth)

print(f"Total questions loaded: {len(all_questions)}")
print(f"Total ground truths loaded: {len(all_answers)}")
print(f"\nSample question: {all_questions[0]}")
print(f"Sample ground truth: {all_answers[0][:300]}...")


Total questions loaded: 6695
Total ground truths loaded: 6695

Sample question: Consider the Non-Disclosure Agreement between CopAcc and ToP Mentors; Does the document indicate that the Agreement does not grant the Receiving Party any rights to the Confidential Information?
Sample ground truth: Any and all proprietary rights, including but not limited to rights to and in inventions, patent rights, utility models, copyrights, trademarks and trade secrets, in and to any Confidential Information shall be and remain with the Participants respectively, and Mentor shall not have any right, licen...


#### **3.1.2** <font color=red> [5 marks] </font>
Create a function to evaluate the generated answers and retrieved contexts.

Evaluate the responses with *Ragas*. Additionally check the retrieval quality using 2 retrieval-driven metrics.

In [1]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics.collections import faithfulness, answer_relevancy, context_recall, context_precision


def evaluate_rag_pipeline(questions, ground_truths):
    """
    Evaluates the RAG pipeline using 10 samples to save time.
    """
    eval_data = {
        "question": [],
        "answer": [],
        "contexts": [],
        "ground_truth": [],
    }

    print(f"Processing {len(questions)} samples...")

    for q, gt in zip(questions, ground_truths):
        # Get retrieval context and generation answer
        docs = retriever.invoke(q)
        eval_data["question"].append(q)
        eval_data["answer"].append(rag_chain.invoke(q))
        eval_data["contexts"].append([d.page_content for d in docs])
        eval_data["ground_truth"].append(gt)

    # Convert to Dataset and evaluate
    ds = Dataset.from_dict(eval_data)
    results = evaluate(
        ds,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
    )

    return results, eval_data

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


#### **3.1.3** <font color=red> [3 marks] </font>
Draw inferences by evaluating answers to questions.

To save time and computing power, you can just run the evaluation on 10 randomly sampled questions.

In [21]:
import random

# Select 10 random samples to save time and credits
sample_indices = random.sample(range(len(all_questions)), 10)
sample_questions = [all_questions[i] for i in sample_indices]
sample_ground_truths = [all_answers[i] for i in sample_indices]

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Run the evaluation using the function defined in 3.1.2
results, eval_data = evaluate_rag_pipeline(sample_questions, sample_ground_truths)

# Display results
print("\nEvaluation Results:")
display(results.to_pandas())

Processing 10 samples...


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[1]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[5]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[9]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')
ERROR:ragas.executor:Exception raised in Job[13]: AttributeError('OpenAIEmbeddings' object has no attribute 'embed_query')


KeyboardInterrupt: 

## **4. Conclusion** <font color=red> [5 marks] </font><br>

### **4.1 Conclusions and insights** <font color=red> [5 marks] </font><br>

#### **4.1.1** <font color=red> [5 marks] </font>
Conclude with the results here. Include the insights gained about the data, model pipeline, the RAG process and the results obtained.